# Reddit Analyzer — EDA Notebook

Exploratory analysis of the collected Reddit database to inform Week 2 training decisions.

**Sections:**
1. Data Volume
2. Text Quality
3. Filtering Preview
4. Vocabulary
5. VADER Distribution
6. Subreddit Sentiment Preview
7. Embedding Sanity Check (MPS-accelerated)

In [ ]:
import sys
sys.path.insert(0, '..')

import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

DB_PATH = '../reddit_data.db'

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
print('Connected to', DB_PATH)

## 1. Data Volume

In [ ]:
post_count = pd.read_sql('SELECT COUNT(*) AS n FROM posts', conn).iloc[0]['n']
comment_count = pd.read_sql('SELECT COUNT(*) AS n FROM comments', conn).iloc[0]['n']

print(f'Posts:    {post_count:,}')
print(f'Comments: {comment_count:,}')
print(f'Total:    {post_count + comment_count:,}')

In [ ]:
# Posts by subreddit
by_sub = pd.read_sql(
    'SELECT subreddit, COUNT(*) AS posts FROM posts GROUP BY subreddit ORDER BY posts DESC LIMIT 25',
    conn
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=by_sub, x='posts', y='subreddit', ax=ax)
ax.set_title('Posts by Subreddit')
ax.set_xlabel('Post Count')
plt.tight_layout()
plt.show()

In [ ]:
# Posts by month
by_month = pd.read_sql(
    "SELECT strftime('%Y-%m', timestamp) AS month, COUNT(*) AS posts "
    "FROM posts WHERE timestamp IS NOT NULL "
    "GROUP BY month ORDER BY month",
    conn
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(by_month['month'], by_month['posts'], marker='o', linewidth=1.5)
ax.set_title('Posts per Month')
ax.set_xlabel('Month')
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 2. Text Quality

In [ ]:
# Sample posts for quality analysis
df_posts = pd.read_sql(
    'SELECT id, title, content, author, subreddit, upvotes FROM posts LIMIT 50000',
    conn
)

df_posts['raw_text'] = (
    df_posts['title'].fillna('') + ' ' + df_posts['content'].fillna('')
).str.strip()
df_posts['text_len'] = df_posts['raw_text'].str.len()
df_posts['is_null_content'] = df_posts['content'].isna() | df_posts['content'].isin(['', '[deleted]', '[removed]'])

print(f'Null/deleted content: {df_posts["is_null_content"].sum():,} ({df_posts["is_null_content"].mean()*100:.1f}%)')
print(f'\nText length percentiles:')
print(df_posts['text_len'].describe(percentiles=[.1, .25, .5, .75, .9, .99]))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
df_posts['text_len'].clip(upper=2000).hist(bins=60, ax=ax, edgecolor='none')
ax.set_title('Raw Text Length Distribution (posts, capped at 2000 chars)')
ax.set_xlabel('Character Count')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Top authors by post volume
top_authors = df_posts['author'].value_counts().head(20)
print('Top 20 authors by post count:')
print(top_authors.to_string())

## 3. Filtering Preview

In [ ]:
import re

_BOT_PATTERN = re.compile(
    r'(?:bot$|automoderator|automod|\[deleted\]|\[removed\]|_bot$|-bot$)',
    re.IGNORECASE
)

df_posts['is_bot'] = df_posts['author'].fillna('').apply(lambda a: bool(_BOT_PATTERN.search(a)))

# Estimate token count from title+content
df_posts['token_count'] = df_posts['raw_text'].str.split().str.len().fillna(0)
df_posts['too_short'] = df_posts['token_count'] < 10

bot_pct = df_posts['is_bot'].mean() * 100
short_pct = (~df_posts['is_bot'] & df_posts['too_short']).mean() * 100
expected_keep = (1 - df_posts['is_bot'].mean() - (~df_posts['is_bot'] & df_posts['too_short']).mean()) * 100

print(f'Bot rate:          {bot_pct:.1f}%')
print(f'Too-short rate:    {short_pct:.1f}%')
print(f'Expected keep:     {expected_keep:.1f}%')
print(f'Estimated clean corpus from {post_count:,} posts: ~{int(post_count * expected_keep / 100):,}')

## 4. Vocabulary

In [ ]:
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords', quiet=True)

STOPWORDS = set(stopwords.words('english'))

sample_texts = df_posts[~df_posts['is_bot'] & ~df_posts['too_short']]['raw_text'].sample(
    min(10000, len(df_posts)), random_state=42
)

words = []
for text in sample_texts:
    words.extend([
        w.lower() for w in re.findall(r'\b[a-z]{3,}\b', text.lower())
        if w not in STOPWORDS
    ])

top_words = Counter(words).most_common(50)
print('Top 50 words (after stopword removal):')
for word, count in top_words:
    print(f'  {word}: {count:,}')

## 5. VADER Distribution

In [ ]:
vader = SentimentIntensityAnalyzer()

sample = df_posts[~df_posts['is_bot'] & ~df_posts['too_short']]['raw_text'].sample(
    min(10000, len(df_posts)), random_state=42
)

compounds = [vader.polarity_scores(t)['compound'] for t in sample]
compounds = pd.Series(compounds)

thresholds = [0.3, 0.4, 0.5, 0.6]
print('Label yield by threshold:')
for t in thresholds:
    kept = (compounds.abs() > t).mean() * 100
    print(f'  |score| > {t}: {kept:.1f}% labeled')

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(compounds, bins=60, edgecolor='none')
ax.axvline(0.5, color='red', linestyle='--', label='threshold=0.5')
ax.axvline(-0.5, color='red', linestyle='--')
ax.set_title('VADER Compound Score Distribution (10k sample)')
ax.set_xlabel('Compound Score')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Subreddit Sentiment Preview

In [ ]:
df_sample = df_posts[~df_posts['is_bot'] & ~df_posts['too_short']].sample(
    min(5000, len(df_posts)), random_state=42
).copy()

df_sample['compound'] = df_sample['raw_text'].apply(
    lambda t: vader.polarity_scores(t)['compound']
)

top_subs = df_sample['subreddit'].value_counts().head(10).index
df_plot = df_sample[df_sample['subreddit'].isin(top_subs)]

fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=df_plot, x='subreddit', y='compound', ax=ax, order=top_subs)
ax.set_title('VADER Compound Score by Subreddit (5k sample)')
ax.set_xlabel('')
ax.set_ylabel('Compound Score')
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 7. Embedding Sanity Check (MPS-accelerated)

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
import umap

device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

sanity_sample = df_posts[~df_posts['is_bot'] & ~df_posts['too_short']].sample(
    min(500, len(df_posts)), random_state=42
)

print(f'Embedding {len(sanity_sample)} samples on {device}...')
embeddings = model.encode(
    sanity_sample['raw_text'].tolist(),
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
)
print(f'Embedding shape: {embeddings.shape}, dtype: {embeddings.dtype}')

In [ ]:
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
coords = reducer.fit_transform(embeddings)

df_viz = sanity_sample[['subreddit']].copy().reset_index(drop=True)
df_viz['x'] = coords[:, 0]
df_viz['y'] = coords[:, 1]

top_subs_embed = df_viz['subreddit'].value_counts().head(8).index
df_viz_filtered = df_viz[df_viz['subreddit'].isin(top_subs_embed)]

fig, ax = plt.subplots(figsize=(10, 7))
for sub in top_subs_embed:
    subset = df_viz_filtered[df_viz_filtered['subreddit'] == sub]
    ax.scatter(subset['x'], subset['y'], label=sub, alpha=0.6, s=15)

ax.set_title('UMAP of 500 Post Embeddings (colored by subreddit)')
ax.legend(markerscale=2, bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

In [ ]:
conn.close()
print('Done.')